# both shoulder
Converted from the latest Python script.


In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import time
import os
from pathlib import Path
from typing import Tuple, List

# --- CONFIGURATION ---
BASE_DIR = Path(__file__).resolve().parent
VIDEO_SOURCE = str(BASE_DIR / "Shoulder_Vids" / "l2.webm")
WINDOW_NAME = "Lateral Raise: Form & Cheat Detector"
SMOOTHING_FACTOR = 0.5
MIN_VISIBILITY = 0.6  # Ignore landmarks below this visibility

# Thresholds
STAGE_DOWN_THRESHOLD = 35
STAGE_UP_THRESHOLD = 80
STREAK_FRAMES_REQUIRED = 3  # Frames required to confirm a transition

# Form Thresholds
ELBOW_BENT_THRESHOLD = 145  # Degrees: Below this is considered too bent
BACK_SWING_THRESHOLD = 20
# Only check elbow form if arm is raised above this angle
CHECK_FORM_HEIGHT_THRESHOLD = 50

# Colors (BGR format)
COLOR_GOOD = (0, 255, 0)  # Green
COLOR_BAD = (0, 0, 255)  # Red
COLOR_WARNING = (0, 255, 255)  # Yellow

mp_pose = mp.solutions.pose


def calculate_angle(a: List, b: List, c: List) -> float:
    """Calculates angle between three points."""
    a = np.array(a)
    b = np.array(b)
    c = np.array(c)
    radians = np.arctan2(c[1] - b[1], c[0] - b[0]) - np.arctan2(
        a[1] - b[1], a[0] - b[0]
    )
    angle = np.abs(radians * 180.0 / np.pi)
    if angle > 180.0:
        angle = 360 - angle
    return angle


def landmarks_visible(lm, indices: List[int]) -> bool:
    """Return True if all requested landmarks are confidently visible."""

    return all(lm[i].visibility >= MIN_VISIBILITY for i in indices)


def get_form_status(
    l_elevation: float,
    r_elevation: float,
    l_elbow_angle: float,
    r_elbow_angle: float,
    torso_angle: float,
) -> Tuple[str, Tuple[int, int, int]]:
    """
    Determine form status.
    Critically: We only check elbow angles if the arms are raised high enough.
    """

    # 1. SWING CHECK (Always active)
    if 180 - torso_angle > BACK_SWING_THRESHOLD:
        return "Don't Swing! Keep Torso Still", COLOR_WARNING

    # 2. ELBOW CHECK (Conditional)
    # Only check elbows if arms are raised > 50 degrees to avoid noise at bottom
    check_left = l_elevation > CHECK_FORM_HEIGHT_THRESHOLD
    check_right = r_elevation > CHECK_FORM_HEIGHT_THRESHOLD

    left_arm_bad = check_left and (l_elbow_angle < ELBOW_BENT_THRESHOLD)
    right_arm_bad = check_right and (r_elbow_angle < ELBOW_BENT_THRESHOLD)

    if left_arm_bad or right_arm_bad:
        return "Straighten Arms!", COLOR_BAD

    # 3. GOOD FORM
    return "Good Form", COLOR_GOOD


def main() -> None:
    # Validate file existence (Optional, good for debugging)
    if VIDEO_SOURCE != 0 and not os.path.exists(VIDEO_SOURCE):
        print(f"Error: Video file not found at {VIDEO_SOURCE}")
        # return  <-- Commented out to allow webcam (0) if file fails

    cap = cv2.VideoCapture(VIDEO_SOURCE)
    if not cap.isOpened():
        print("Error reading video source.")
        return

    # FPS Calculation
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps == 0 or fps > 120:
        fps = 30
    delay = int(1000 / fps)

    # -- COUNTER VARIABLES --
    reps = 0
    stage = "down"
    hold_start = 0
    current_hold_time = 0.0

    # -- SMOOTHING VARIABLES --
    prev_l_elevation = 0
    prev_r_elevation = 0
    prev_l_form = 180  # Default to straight
    prev_r_form = 180
    prev_torso = 180

    # -- STATE SAFEGUARDS --
    l_elevation = r_elevation = 0.0
    l_form = r_form = 180.0
    torso_angle = 180.0

    up_streak = 0
    down_streak = 0

    with mp_pose.Pose(
        min_detection_confidence=0.6, min_tracking_confidence=0.6
    ) as pose:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
                continue

            # Resize
            frame = cv2.resize(frame, (900, 600))

            # UI Canvas
            final_image = np.zeros((600, 1300, 3), dtype=np.uint8)
            final_image[0:600, 0:900] = frame

            # Process Pose
            h, w, _ = frame.shape
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = pose.process(rgb)

            feedback = "Resting"
            color_status = (200, 200, 200)

            if results.pose_landmarks:
                lm = results.pose_landmarks.landmark

                visibility_ok = landmarks_visible(lm, [11, 12, 13, 14, 15, 16, 23, 24])

                if visibility_ok:
                    # --- 1. EXTRACT POINTS ---
                    l_shoulder = [lm[11].x * w, lm[11].y * h]
                    l_elbow = [lm[13].x * w, lm[13].y * h]
                    l_wrist = [lm[15].x * w, lm[15].y * h]
                    l_hip = [lm[23].x * w, lm[23].y * h]

                    r_shoulder = [lm[12].x * w, lm[12].y * h]
                    r_elbow = [lm[14].x * w, lm[14].y * h]
                    r_wrist = [lm[16].x * w, lm[16].y * h]
                    r_hip = [lm[24].x * w, lm[24].y * h]

                    # --- 2. CALCULATE ANGLES ---
                    raw_l_elev = calculate_angle(l_hip, l_shoulder, l_elbow)
                    raw_r_elev = calculate_angle(r_hip, r_shoulder, r_elbow)
                    raw_l_form = calculate_angle(l_shoulder, l_elbow, l_wrist)
                    raw_r_form = calculate_angle(r_shoulder, r_elbow, r_wrist)

                    # Torso
                    mid_hip = [
                        (l_hip[0] + r_hip[0]) / 2,
                        (l_hip[1] + r_hip[1]) / 2,
                    ]
                    mid_shldr = [
                        (l_shoulder[0] + r_shoulder[0]) / 2,
                        (l_shoulder[1] + r_shoulder[1]) / 2,
                    ]
                    vertical_point = [mid_hip[0], mid_hip[1] + 100]
                    raw_torso = calculate_angle(mid_shldr, mid_hip, vertical_point)

                    # --- 3. SMOOTHING ---
                    # Initialize previous values if first frame
                    if prev_l_elevation == 0:
                        prev_l_elevation = raw_l_elev
                        prev_r_elevation = raw_r_elev
                        prev_l_form = raw_l_form
                        prev_r_form = raw_r_form
                        prev_torso = raw_torso

                    l_elevation = (prev_l_elevation * (1 - SMOOTHING_FACTOR)) + (
                        raw_l_elev * SMOOTHING_FACTOR
                    )
                    r_elevation = (prev_r_elevation * (1 - SMOOTHING_FACTOR)) + (
                        raw_r_elev * SMOOTHING_FACTOR
                    )
                    l_form = (prev_l_form * (1 - SMOOTHING_FACTOR)) + (
                        raw_l_form * SMOOTHING_FACTOR
                    )
                    r_form = (prev_r_form * (1 - SMOOTHING_FACTOR)) + (
                        raw_r_form * SMOOTHING_FACTOR
                    )
                    torso_angle = (prev_torso * (1 - SMOOTHING_FACTOR)) + (
                        raw_torso * SMOOTHING_FACTOR
                    )

                    prev_l_elevation = l_elevation
                    prev_r_elevation = r_elevation
                    prev_l_form = l_form
                    prev_r_form = r_form
                    prev_torso = torso_angle

                    # --- 4. CHECK FORM (Logic Updated) ---
                    feedback, color_status = get_form_status(
                        l_elevation, r_elevation, l_form, r_form, torso_angle
                    )

                    # --- 5. REP COUNTING (Debounced to avoid flicker) ---
                    if (
                        l_elevation > STAGE_UP_THRESHOLD
                        and r_elevation > STAGE_UP_THRESHOLD
                    ):
                        up_streak += 1
                        down_streak = 0
                        if stage == "down" and up_streak >= STREAK_FRAMES_REQUIRED:
                            stage = "up"
                            hold_start = time.time()
                        current_hold_time = time.time() - hold_start

                    elif (
                        l_elevation < STAGE_DOWN_THRESHOLD
                        and r_elevation < STAGE_DOWN_THRESHOLD
                    ):
                        down_streak += 1
                        up_streak = 0
                        if stage == "up" and down_streak >= STREAK_FRAMES_REQUIRED:
                            stage = "down"
                            reps += 1
                            current_hold_time = 0.0
                    else:
                        up_streak = 0
                        down_streak = 0

                    # --- 6. DRAWING ---
                    # Draw Body
                    for p1, p2 in [
                        (l_shoulder, l_hip),
                        (r_shoulder, r_hip),
                        (l_shoulder, l_elbow),
                        (l_elbow, l_wrist),
                        (r_shoulder, r_elbow),
                        (r_elbow, r_wrist),
                    ]:
                        cv2.line(
                            final_image,
                            tuple(np.array(p1, int)),
                            tuple(np.array(p2, int)),
                            (255, 255, 255),
                            2,
                        )

                    # Draw Joints (Color based on status)
                    for pt in [l_elbow, r_elbow, l_shoulder, r_shoulder]:
                        cv2.circle(
                            final_image,
                            tuple(np.array(pt, int)),
                            6,
                            color_status,
                            -1,
                        )
                else:
                    feedback = "Move fully into frame"
                    color_status = COLOR_WARNING

            # --- STATS PANEL ---
            cv2.putText(
                final_image,
                "REPS:",
                (930, 60),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (255, 255, 255),
                2,
            )
            cv2.putText(
                final_image,
                str(reps),
                (930, 120),
                cv2.FONT_HERSHEY_SIMPLEX,
                2.0,
                color_status,
                3,
            )

            cv2.putText(
                final_image,
                f"STAGE: {stage.upper()}",
                (930, 170),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (255, 255, 255),
                2,
            )

            cv2.putText(
                final_image,
                f"HOLD: {current_hold_time:0.1f}s",
                (930, 210),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (180, 180, 180),
                1,
            )

            # Show Metrics
            cv2.putText(
                final_image,
                f"L ELBOW: {int(l_form)}",
                (930, 300),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (255, 255, 255),
                1,
            )
            cv2.putText(
                final_image,
                f"R ELBOW: {int(r_form)}",
                (930, 330),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (255, 255, 255),
                1,
            )
            cv2.putText(
                final_image,
                f"L ELEV:  {int(l_elevation)}",
                (930, 360),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (255, 255, 255),
                1,
            )

            # Show Status
            cv2.putText(
                final_image,
                "STATUS:",
                (930, 500),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (255, 255, 255),
                2,
            )
            cv2.putText(
                final_image,
                feedback,
                (925, 550),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                color_status,
                2,
            )

            cv2.imshow(WINDOW_NAME, final_image)
            if cv2.waitKey(delay) & 0xFF == ord("q"):
                break

    cap.release()
    cv2.destroyAllWindows()


if __name__ == "__main__":
    main()
